# InternSpark Task 1 — ML Classification Project

**Dataset:** Breast Cancer Wisconsin Diagnostic dataset (scikit-learn)

**Models:** Logistic Regression and Random Forest

This notebook covers preprocessing, train/test split, 5-fold cross-validation, model comparison, and accuracy/precision/recall/F1/ROC-AUC metrics.

In [ ]:

# InternSpark Task 1 — ML Classification Project
# Breast Cancer Classification using Logistic Regression and Random Forest

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

# 1. Load dataset
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print("Dataset shape:", X.shape)
print("Classes:", dict(enumerate(data.target_names)))
display(X.head())

# 2. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# 3. Models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    )
}

# 4. Five-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc"}

cv_table = []
for name, model in models.items():
    cv_result = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    for metric in scoring:
        cv_table.append([
            name, metric.upper().replace("_","-"),
            cv_result[f"test_{metric}"].mean(),
            cv_result[f"test_{metric}"].std()
        ])

cv_df = pd.DataFrame(cv_table, columns=["Model","Metric","CV Mean","CV Std"])
display(cv_df)

# 5. Fit and evaluate on held-out test set
test_rows = []
fitted = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]
    test_rows.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred),
        recall_score(y_test, pred),
        f1_score(y_test, pred),
        roc_auc_score(y_test, prob)
    ])

test_df = pd.DataFrame(
    test_rows,
    columns=["Model","Accuracy","Precision","Recall","F1","ROC-AUC"]
)
display(test_df)

# 6. Classification reports and confusion matrices
for name, model in fitted.items():
    pred = model.predict(X_test)
    print("\n", name)
    print(classification_report(y_test, pred, target_names=data.target_names))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, pred))

# 7. ROC curves
plt.figure(figsize=(7,5))
for name, model in fitted.items():
    prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()

# 8. Conclusion
print("Model comparison completed.")
print("The model with the higher test ROC-AUC/F1 can be selected based on the project objective.")
